# HYDROSIGHT — Water Main Breaks EDA

**Calgary · Historical water main break observations**

This notebook documents the original analyses, cleaning rules, limitations and methodological decisions. All computations and visualizations run directly in Colab. Only the cleaned JSONL dataset is exported.

## 1. Source data

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT_DIR = Path("/content/HYDROSIGHT")
if not ROOT_DIR.exists():
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/cpoiato/HYDROSIGHT.git", str(ROOT_DIR)], check=True)

RAW_DIR = ROOT_DIR / "data" / "raw" / "water_breaks"
source_files = sorted(RAW_DIR.glob("water_main_breaks_*.jsonl"))
assert source_files, f"No source snapshots found in {RAW_DIR}"
source_file = source_files[-1]
df = pd.read_json(source_file, lines=True)
print("Source:", source_file)
print("Original records:", len(df))
display(df.head())

## 2. Data quality and coordinate conversion

Convert the recorded break date and GeoJSON point into typed columns. Inspect missing values, invalid dates and coordinates before cleaning. The approximate Calgary bounding box is a **diagnostic**, not an official municipal boundary.

In [ ]:
df = df.copy()
df["break_date"] = pd.to_datetime(df["break_date"], errors="coerce")

point_coordinates = df["point"].map(
    lambda point: point.get("coordinates") if isinstance(point, dict) else None
)
df["longitude"] = pd.to_numeric(
    point_coordinates.map(lambda coords: coords[0] if isinstance(coords, (list, tuple)) and len(coords) >= 2 else np.nan),
    errors="coerce"
)
df["latitude"] = pd.to_numeric(
    point_coordinates.map(lambda coords: coords[1] if isinstance(coords, (list, tuple)) and len(coords) >= 2 else np.nan),
    errors="coerce"
)
df = df.drop(columns=["point"])

print("=== DATASET INFORMATION ===")
df.info()
print("\n=== MISSING VALUES ===")
display(df.isna().sum().rename("missing_count").to_frame())

invalid_date = df["break_date"].isna()
future_date = df["break_date"] > pd.Timestamp.now()
valid_global = df["longitude"].between(-180, 180) & df["latitude"].between(-90, 90)
calgary_bounds = df["longitude"].between(-114.5, -113.5) & df["latitude"].between(50.5, 51.5)

print("Invalid dates:", int(invalid_date.sum()))
print("Future dates:", int(future_date.sum()))
print("Invalid global coordinates:", int((~valid_global).sum()))
print("Outside approximate Calgary bounds:", int((valid_global & ~calgary_bounds).sum()))
print("Earliest date:", df["break_date"].min())
print("Latest date:", df["break_date"].max())
display(df.loc[df.isna().any(axis=1)].head(20))

## 3. Cleaning — original policy retained

The source implementation removes **incomplete records** and keeps one record per duplicate group defined by `break_date`, `break_type`, `status`, `longitude`, and `latitude`.

**Limitation:** identical values across these five attributes suggest probable duplicates but do not establish identical events without a unique event identifier. Removing records with missing `break_type` can also discard real ruptures. This notebook deliberately **preserves the original cleaning policy**, rather than silently changing the study sample. The raw snapshot remains unchanged.

In [ ]:
DUPLICATE_COLUMNS = ["break_date", "break_type", "status", "longitude", "latitude"]
original_count = len(df)
incomplete = df.isna().any(axis=1)
missing_removed = int(incomplete.sum())
missing_in_2016_2025 = int((incomplete & df["break_date"].dt.year.between(2016, 2025)).sum())

df_clean = df.dropna().copy()
duplicate_groups = df_clean.loc[
    df_clean.duplicated(subset=DUPLICATE_COLUMNS, keep=False)
].sort_values(DUPLICATE_COLUMNS)
duplicate_excess = int(df_clean.duplicated(subset=DUPLICATE_COLUMNS, keep="first").sum())

print("=== DUPLICATE GROUPS ===")
print("Records involved:", len(duplicate_groups))
print("Excess records:", duplicate_excess)
display(duplicate_groups.head(30))

df_clean = df_clean.drop_duplicates(subset=DUPLICATE_COLUMNS, keep="first").copy()
print("=== CLEANING SUMMARY ===")
print("Original records:", original_count)
print("Incomplete removed:", missing_removed)
print("Incomplete removed within 2016–2025:", missing_in_2016_2025)
print("Probable duplicates removed:", duplicate_excess)
print("Clean records:", len(df_clean))
assert len(df_clean) == original_count - missing_removed - duplicate_excess

## 4. Descriptive and categorical distributions

Review numerical summaries and the distributions of `break_type` and `status`. The raw category codes and `ACTIVE`/`RETIRED` status labels must **not** be assigned meanings absent an official data dictionary.

In [ ]:
display(df_clean.describe(include="all"))

break_type_counts = df_clean["break_type"].value_counts(dropna=False)
status_counts = df_clean["status"].value_counts(dropna=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
break_type_counts.plot.bar(ax=axes[0], title="Water main breaks by type")
status_counts.plot.bar(ax=axes[1], title="Water main breaks by status")
axes[0].set_xlabel("Source break type code")
axes[1].set_xlabel("Source status code")
for ax in axes:
    ax.set_ylabel("Break count")
    ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
plt.show()

## 5. Annual and monthly patterns

Annual and monthly rupture counts are inspected across the available history. The IQR rule highlights unusual periods **for investigation only**; outlier years and months are not removed. Zero-event months between the first and last complete-year observations are included in the monthly series.

In [ ]:
annual_counts = df_clean["break_date"].dt.year.value_counts().sort_index()
complete_years = df_clean.loc[df_clean["break_date"].dt.year < datetime.now().year].copy()
annual_complete = complete_years["break_date"].dt.year.value_counts().sort_index()

q1, q3 = annual_complete.quantile([0.25, 0.75])
annual_low, annual_high = max(0, q1 - 1.5 * (q3 - q1)), q3 + 1.5 * (q3 - q1)
annual_outliers = annual_complete[(annual_complete < annual_low) | (annual_complete > annual_high)]

month_counts = complete_years["break_date"].dt.to_period("M").value_counts().sort_index()
if len(month_counts):
    month_counts = month_counts.reindex(pd.period_range(month_counts.index.min(), month_counts.index.max(), freq="M"), fill_value=0)
monthly_q1, monthly_q3 = month_counts.quantile([0.25, 0.75])
monthly_low, monthly_high = max(0, monthly_q1 - 1.5 * (monthly_q3 - monthly_q1)), monthly_q3 + 1.5 * (monthly_q3 - monthly_q1)
monthly_outliers = month_counts[(month_counts < monthly_low) | (month_counts > monthly_high)]

print("=== ANNUAL IQR OUTLIERS (FLAG ONLY) ===")
display(annual_outliers.rename("breaks").to_frame())
print("=== MONTHLY IQR OUTLIERS (FLAG ONLY) ===")
display(monthly_outliers.rename("breaks").to_frame())

fig, ax = plt.subplots(figsize=(14, 5))
annual_counts.plot.bar(ax=ax)
ax.set(title="Annual water main breaks — historical", xlabel="Year", ylabel="Break count")
ax.tick_params(axis="x", labelrotation=90, labelsize=7)
fig.tight_layout()
plt.show()

## 6. Seasonality and date precision

Meteorological seasons: winter (December–February), spring (March–May), summer (June–August), autumn (September–November).

The day-of-month distribution is a **date-quality diagnostic**, not a daily seasonality measure: day 31 naturally has fewer opportunities than day 1. A high concentration on the first day of a month may indicate approximate or administratively assigned dates, but the cause cannot be proven from the histogram.

In [ ]:
year_month = complete_years.groupby([
    complete_years["break_date"].dt.year.rename("year"),
    complete_years["break_date"].dt.month.rename("month")
]).size()
years = sorted(complete_years["break_date"].dt.year.unique())
month_grid = pd.MultiIndex.from_product([years, range(1, 13)], names=["year", "month"])
monthly_average = year_month.reindex(month_grid, fill_value=0).groupby(level="month").mean()

month = df_clean["break_date"].dt.month
season_labels = np.select(
    [month.isin([12, 1, 2]), month.isin([3, 4, 5]), month.isin([6, 7, 8]), month.isin([9, 10, 11])],
    ["Winter", "Spring", "Summer", "Autumn"], default="Unknown"
)
season_counts = pd.Series(season_labels, index=df_clean.index).value_counts().reindex(["Winter", "Spring", "Summer", "Autumn"], fill_value=0)
day_counts = df_clean["break_date"].dt.day.value_counts().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
monthly_average.plot.bar(ax=axes[0], title="Average breaks by month")
season_counts.plot.bar(ax=axes[1], title="Breaks by meteorological season")
day_counts.plot.bar(ax=axes[2], title="Breaks by day of month")
for ax in axes:
    ax.set_ylabel("Break count")
fig.tight_layout()
plt.show()

annual_total = df_clean["break_date"].dt.year.value_counts().sort_index()
first_day = df_clean.loc[df_clean["break_date"].dt.day.eq(1), "break_date"].dt.year.value_counts().reindex(annual_total.index, fill_value=0)
first_day_share = first_day.div(annual_total).mul(100)
fig, ax = plt.subplots(figsize=(14, 4))
first_day_share.plot.bar(ax=ax)
ax.set(title="Breaks recorded on the first day of a month", xlabel="Year", ylabel="Share (%)")
ax.tick_params(axis="x", labelrotation=90, labelsize=7)
fig.tight_layout()
plt.show()

### Interpretation — Historical date quality

An unusually high concentration of records dated on the first of a month was observed in older historical periods, especially before 2000. The exact reason remains unknown. Because uncertain daily dates could misalign ruptures with daily meteorological observations, **pre-2000 events are retained in the historical cleaned dataset but excluded from the principal modeling scope**. Historical comparisons remain descriptive and are not evidence of a causal weather relationship.

## 7. High-break years: 1982–1986

The original EDA identified elevated break counts between 1982 and 1986, with winter peaks in several years and a notable October peak in 1986. These patterns may reflect weather, reporting practices, infrastructure conditions or other factors. They must not be treated as proof of a temperature-related causal mechanism.

In [ ]:
high_break_period = df_clean.loc[df_clean["break_date"].dt.year.between(1982, 1986)]
monthly_1982_1986 = high_break_period.groupby([
    high_break_period["break_date"].dt.month.rename("month"),
    high_break_period["break_date"].dt.year.rename("year")
]).size().unstack("year", fill_value=0).reindex(range(1, 13), fill_value=0)

display(monthly_1982_1986)
fig, ax = plt.subplots(figsize=(12, 5))
for year in monthly_1982_1986.columns:
    ax.plot(monthly_1982_1986.index, monthly_1982_1986[year], marker="o", label=str(year))
ax.set(title="Monthly water main breaks — 1982–1986", xlabel="Month", ylabel="Break count", xticks=range(1, 13))
ax.legend(title="Year")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 8. Spatial distribution

Plot the full historical rupture locations and the 2016–2025 subset. The plots describe recorded events, not exposure-adjusted rupture risk. A current pipe inventory must not be assumed to represent the historic network for every year.

In [ ]:
import geopandas as gpd

recent = df_clean.loc[df_clean["break_date"].dt.year.between(2016, 2025)].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, data, title in [
    (axes[0], df_clean, "Recorded breaks — complete history"),
    (axes[1], recent, "Recorded breaks — 2016–2025")
]:
    spatial = data.dropna(subset=["longitude", "latitude"])
    if not spatial.empty:
        geo = gpd.GeoDataFrame(spatial, geometry=gpd.points_from_xy(spatial["longitude"], spatial["latitude"]), crs="EPSG:4326")
        geo.plot(ax=ax, markersize=2, alpha=0.4)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
fig.tight_layout()
plt.show()
print("Historical spatial observations:", len(df_clean))
print("2016–2025 spatial observations:", len(recent))

## 9. Focused study period: 2000–2025

Repeat the key descriptive views for 2000–2025 to reduce the influence of older records with uncertain day-level date precision. This exploratory scope is **not** the same as the current **2016–2025 prediction/evaluation scope**; selection of modeling years must be made explicitly in the modeling pipeline.

In [ ]:
df_study = df_clean.loc[df_clean["break_date"].dt.year.between(2000, 2025)].copy()
study_annual = df_study["break_date"].dt.year.value_counts().sort_index()
study_monthly_grid = pd.MultiIndex.from_product([range(2000, 2026), range(1, 13)], names=["year", "month"])
study_monthly = df_study.groupby([
    df_study["break_date"].dt.year.rename("year"),
    df_study["break_date"].dt.month.rename("month")
]).size().reindex(study_monthly_grid, fill_value=0).groupby(level="month").mean()
study_month = df_study["break_date"].dt.month
study_seasons = pd.Series(np.select(
    [study_month.isin([12, 1, 2]), study_month.isin([3, 4, 5]), study_month.isin([6, 7, 8]), study_month.isin([9, 10, 11])],
    ["Winter", "Spring", "Summer", "Autumn"], default="Unknown"
), index=df_study.index).value_counts().reindex(["Winter", "Spring", "Summer", "Autumn"], fill_value=0)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
study_annual.plot.bar(ax=axes[0, 0], title="Annual breaks — 2000–2025")
study_monthly.plot.bar(ax=axes[0, 1], title="Monthly average — 2000–2025")
study_seasons.plot.bar(ax=axes[1, 0], title="Seasonal breaks — 2000–2025")
df_study["break_type"].value_counts().plot.bar(ax=axes[1, 1], title="Break type codes — 2000–2025")
for ax in axes.flat:
    ax.set_ylabel("Break count")
    ax.tick_params(axis="x", labelrotation=70)
fig.tight_layout()
plt.show()
print("Study-period records (2000–2025):", len(df_study))

## 10. Methodological decisions

- The full cleaned historical dataset is retained; **pre-2000 events are excluded only from the principal temporal modeling scope** because of uncertain day-level dates.
- Unusually high annual/monthly counts are flagged, **not removed**.
- Missing-field removal and five-field probable deduplication are preserved from the original EDA, with the explicit caveat that genuine events may have been excluded.
- Categorical codes are shown without assigning unsupported meanings.
- Winter seasonality and 1982–1986 peaks are descriptive observations; a relationship with weather must be tested using aligned historical weather data.
- The planned model uses the **2016–2025** period, with spatial and temporal validation handled separately; the 2000–2025 plots remain historical context.

## 11. Cleaned historical dataset — JSONL export

In [ ]:
output_dir = ROOT_DIR / "data" / "processed" / "water_breaks"
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_file = output_dir / f"water_main_breaks_clean_{timestamp}.jsonl"

assert df_clean["break_date"].notna().all()
assert not df_clean.duplicated(subset=DUPLICATE_COLUMNS).any()
assert len(df_clean) == original_count - missing_removed - duplicate_excess

df_clean.to_json(output_file, orient="records", lines=True, date_format="iso", force_ascii=False)
print("=== CLEAN WATER MAIN BREAKS EXPORTED ===")
print("File:", output_file)
print("Records:", len(df_clean))
print("Original source preserved:", source_file)